# EfficientNet Training Guide: Placenta Segmentation with Image Augmentation

This notebook provides a complete workflow for:
1. **Augmenting your image dataset** to increase training examples
2. **Training an EfficientNet model** for 3-class placenta segmentation
3. **Evaluating and visualizing results**

## Dataset Structure
- **Classes**: 0 (Background), 1 (Fetal), 2 (Maternal)
- **Masks**: Single-channel grayscale PNG files (values 0, 1, 2)
- **Format**: Training images in `data/images/`, masks in `data/masks/`

## Part 1: Import Required Libraries

In [ ]:
import os
import sys
import cv2
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import logging

# Add project to path
project_root = Path('.').resolve()
sys.path.insert(0, str(project_root))

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Check CUDA availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")
if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"CUDA Version: {torch.version.cuda}")

## Part 2: Quick Image Augmentation

Let's augment your dataset first to increase training examples. This will improve model generalization.

In [ ]:
# Option 1: Use the built-in augmentation utility from project
# This runs augmentation using the augment_images function from utilities/image_augmentation.py

import subprocess

print("=" * 60)
print("RUNNING IMAGE AUGMENTATION")
print("=" * 60)

# Run augmentation with 3 repetitions per image
# This will create 3x more training data
augment_cmd = [
    "python", "-m", "utilities.image_augmentation",
    "--repetitions", "3",
    "--prefix", "augmented"
]

logger.info(f"Running augmentation: {' '.join(augment_cmd)}")
result = subprocess.run(augment_cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("Warnings/Errors:", result.stderr)

# Verify augmented data was created
# Check for both TIF and PNG images
images_dir = Path("data/images")
masks_dir = Path("data/masks")

if images_dir.exists():
    # Count both TIF and PNG files
    num_tif_images = len(list(images_dir.glob("*.TIF")))
    num_png_images = len(list(images_dir.glob("*.png")))
    num_total = num_tif_images + num_png_images
    
    logger.info(f"Total images in data/images/: {num_total}")
    logger.info(f"  - TIF files: {num_tif_images}")
    logger.info(f"  - PNG files: {num_png_images}")
else:
    logger.warning("Images directory not found")

## Part 3: Train EfficientNet Model

In [ ]:
print("=" * 60)
print("STARTING EFFICIENTNET TRAINING")
print("=" * 60)

training_cmd = [
    "python", "-m", "models.efficicentnet_train_smp",
    "--epochs", "50",
    "--batch-size", "16",  # A40 can handle batch size 16 (10x faster than batch size 1)
    "--augment",  # Enable on-the-fly augmentation
    "--early-stopping-patience", "10",  # Stop if validation loss doesn't improve
    "--lr-patience", "5",
    "--lr-factor", "0.5"
]

logger.info(f"Training command: {' '.join(training_cmd)}")
print("\nStarting training... This will take ~5-10 minutes with A40 GPU.")
print("Monitor GPU usage with: nvidia-smi")
print()

result = subprocess.run(training_cmd, capture_output=False, text=True)

if result.returncode != 0:
    logger.error(f"Training failed with return code {result.returncode}")
else:
    logger.info("Training completed successfully!")
    
    # Check if model was saved
    model_best = Path("trained_models/efficientnet_unet_placenta_best.pth")
    model_final = Path("trained_models/efficientnet_unet_placenta_final.pth")
    
    if model_best.exists():
        size_mb = model_best.stat().st_size / (1024 * 1024)
        logger.info(f"Best model saved: {model_best} ({size_mb:.2f} MB)")
    
    if model_final.exists():
        size_mb = model_final.stat().st_size / (1024 * 1024)
        logger.info(f"Final model saved: {model_final} ({size_mb:.2f} MB)")

## Part 4: Test on a Single Image

Let's test the trained model on a single image to verify it's working.

**Note**: If you have validation images, use one of those. Otherwise, the model will test on a training image.

In [ ]:
# Find a test image (look for both TIF and PNG)
images_dir = Path("data/images")
image_files = list(images_dir.glob("*.TIF")) + list(images_dir.glob("*.png"))

if not image_files:
    logger.error("No images found in data/images/")
else:
    # Pick the first image for testing
    test_image_path = str(image_files[0])
    logger.info(f"Testing with image: {test_image_path}")
    
    # Use the inference utility
    inference_cmd = [
        "python", "-m", "utilities.inference",
        "--image", test_image_path,
        "--architecture", "efficientnet",
        "--draw-contours",
        "--verbose"
    ]
    
    logger.info(f"Running inference: {' '.join(inference_cmd)}")
    result = subprocess.run(inference_cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print("Output:", result.stderr)

## Quick Reference: Running Commands from Terminal

If you prefer to run these commands directly from your terminal instead of the notebook, use these commands:

In [ ]:
commands = """
# === REQUIRED: SET CUDA PATHS (for cluster) ===
export CUDA_HOME=/usr/local/lib/ollama/cuda_v12
export PATH=$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME:$LD_LIBRARY_PATH

# === STEP 1: AUGMENT DATA ===
# Augment images 3x (creates 3 variations per original image)
python -m utilities.image_augmentation --repetitions 3 --prefix augmented

# === STEP 2: TRAIN EFFICIENTNET (A40 OPTIMIZED) ===
# Optimized for A40 GPU: batch size 16, mixed precision, early stopping
python -m models.efficicentnet_train_smp \\
    --epochs 50 \\
    --batch-size 16 \\
    --augment \\
    --early-stopping-patience 10 \\
    --lr-patience 5 \\
    --lr-factor 0.5

# === STEP 3: TEST ON SINGLE IMAGE ===
# Test on a TIF image
python -m utilities.inference \\
    --image data/images/100.TIF \\
    --architecture efficientnet \\
    --draw-contours \\
    --verbose

# === STEP 4: TRAIN ALL MODELS ===
# Train all 4 architectures (EfficientNet, U-Net, RegNet, ViT)
python -m models.train_all_models \\
    --models efficientnet unet regnet vit \\
    --epochs 50 \\
    --batch-size 16

# === MONITORING ===
# Watch GPU usage (run in another terminal):
nvidia-smi -l 1  # Refreshes every 1 second

# Check disk usage:
du -sh trained_models/  # Size of saved models
du -sh data/            # Size of data directory

# === PARAMETERS EXPLANATION ===
# --batch-size 16   : A40 has 48GB VRAM, can handle batch size 16 (10x faster)
# --augment         : Enable on-the-fly augmentation (different per epoch)
# --early-stopping-patience 10 : Stop if validation loss doesn't improve for 10 epochs
# --epochs 50       : Maximum 50 epochs (may stop earlier with early stopping)
"""

print(commands)

# Save to file for reference
with open("TERMINAL_COMMANDS.txt", "w") as f:
    f.write(commands)
    
logger.info("Commands saved to TERMINAL_COMMANDS.txt")

## Troubleshooting & Tips

### Common Issues

**Issue: Out of Memory (OOM) Error**
- Solution: Reduce batch size (--batch-size 2 or 1)
- Or reduce number of epochs

**Issue: Model not training (loss not decreasing)**
- Check that images and masks are aligned
- Verify mask values are 0, 1, 2 (not floating point)
- Check image dimensions match expected input

**Issue: Augmentation not creating files**
- Verify data/images/ and data/masks/ directories exist
- Make sure files have .png extension
- Check write permissions on data directory

### Optimization Tips

1. **For better results**: Use more augmentation repetitions (5-10x)
2. **For faster training**: Reduce image resolution or batch size
3. **For production**: Train multiple epochs and select best checkpoint
4. **For better accuracy**: Use ensemble of multiple models

## Workflow Summary

```
┌─────────────────────┐
│   Raw Dataset       │  (images/ and masks/ folders)
└──────────┬──────────┘
           │
           ▼
┌─────────────────────────────────────────┐
│  Step 1: AUGMENT DATASET                │
│  python -m utilities.image_augmentation │
│  --repetitions 3                        │
└──────────┬──────────────────────────────┘
           │
           ▼  (3-10x more training data)
┌──────────────────────────────────────────────┐
│  Step 2: TRAIN EFFICIENTNET                  │
│  python -m models.efficicentnet_train_smp    │
│  --epochs 50 --batch-size 4                  │
└──────────┬───────────────────────────────────┘
           │
           ▼  (validates on 20% holdout)
┌─────────────────────────────┐
│  Trained Model Saved        │
│  trained_models/            │
│  efficientnet_unet_*.pth    │
└──────────┬──────────────────┘
           │
           ▼
┌──────────────────────────────────────────┐
│  Step 3: TEST INFERENCE                  │
│  python -m utilities.inference           │
│  --image data/images/100.png             │
└──────────────────────────────────────────┘
```

## Next Steps

1. **Run the notebook cells in order** (or use terminal commands)
2. **Monitor training**: Watch loss decrease and validation metrics improve
3. **Test results**: Check predicted segmentation masks visually
4. **Iterate**: Adjust hyperparameters if needed (batch size, epochs, learning rate)
5. **Compare models**: Train other architectures (U-Net, RegNet, ViT) for comparison